In [4]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from bct.utils.other import threshold_proportional, normalize
from small_world_propensity import SWP, characteristic_path_length, clustering_coefficient_bct
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler

In [2]:
fc = np.load('/home/gbz6qn/Documents/research/data/coupling/working/included_subs/spis/100206/symmetrized/covariance_symmetrized.npy')
sc = np.load('/home/gbz6qn/Documents/research/data/hcp_shen_sc/sub-100206_parc-shen268_tract-prob_sc.npy')

In [3]:
spi_dir = '/home/gbz6qn/Documents/research/data/coupling/working/included_subs/spis/'
sc_dir = '/home/gbz6qn/Documents/research/data/hcp_shen_sc/'
# List of subs
with open('/home/gbz6qn/Documents/research/code/coupling/compute_gca/subs.txt', 'r') as f:
    lines = f.read()
    subs = lines.split('\n')[:-1]
subs = sorted(subs)

## Thresholding by only retaining edges that also exist in the structural connectivity matrix

In [46]:
SWP_cov = []
delta_C_cov = []
delta_L_cov = []

for i, sub in tqdm(enumerate(subs)):
    fc_path = os.path.join(spi_dir, sub, 'symmetrized', 'covariance_symmetrized.npy')
    sc_path = os.path.join(sc_dir, f'sub-{sub}_parc-shen268_tract-prob_sc.npy')

    fc = np.load(fc_path)
    sc = np.load(sc_path)

    sparser = fc*(sc!=0)
    swp, delta_C, delta_L = SWP(sparser)
    SWP_cov.append(swp)
    delta_C_cov.append(delta_C)
    delta_L_cov.append(delta_L)

SWP_fc = np.array(SWP_cov)
SWP_fc = np.mean(SWP_fc[~np.isnan(SWP_fc)])
delta_C_fc = np.array(delta_C_cov)
delta_C_fc = np.mean(delta_C_fc[~np.isnan(delta_C_fc)])
delta_L_fc = np.array(delta_L_cov)
delta_L_fc = np.mean(delta_L_fc[~np.isnan(delta_L_fc)])

329it [03:53,  1.41it/s]


In [47]:
print(f'SWP: {SWP_fc:.3f}, delta_C: {delta_C_fc:.3f}, delta_L: {delta_L_fc}')

SWP: 0.585, delta_C: 0.311, delta_L: 0.4496801512197063


## Threshold using proportional thresholding at various levels
I want to examine how clustering, path length, and SWP change at the various levels